# 17. SQL 에이전트 빌드 — LangGraph StateGraph (★ 과정의 정점)
> Day 3 · 20H · 소요 약 50분 + 본인 프로젝트 적용

## 학습 목표

- **AgentState** (TypedDict) 스키마를 설계한다 — `question`, `sql`, `result`, `answer`, `error`, `retry_count`.
- **4개 노드**를 직접 구현한다:
  1. `generate_sql` — 질문 → 스키마 프롬프트 → 원시 SQL.
  2. `execute_sql` — SQLAlchemy 로 읽기 전용 실행, 에러 캡처.
  3. `validate_sql` — 위험 키워드 차단, LIMIT 강제.
  4. `generate_answer` — 결과 → 한국어 자연어 요약.
- **조건부 재시도 루프** — 에러 발생 시 `retry_count < 3` 조건으로 `generate_sql` 로 되돌아가 **이전 에러를 프롬프트에 피드백**하여 재생성.
- `stream()` 으로 상태 전이를 추적하고, 10 개 질문 일괄 테스트로 **재시도가 실제 발동**하는 모습을 관찰한다.
- **본인 프로젝트 DB 로 전환** 하는 체크리스트 (DSN/TABLES/프롬프트 규칙 3 곳만 수정).

## 이 노트북의 의미

> Day 1 SQL → Day 2 Text-to-SQL → Day 3 LangChain/LangGraph — **모든 재료가 여기서 합쳐집니다.**
> 이 노트북의 에이전트가 **과제 #3 의 기반** 이며, Day 4 에서 LangSmith 트레이싱 + Ragas 평가를 붙여 최종 발표용 에이전트가 됩니다.

> **자기완결성 원칙.** 14/15/16 을 건너뛰고 본 노트북만 열어도 동작합니다. Neon 병원 DB (Day 1 01~03 번에서 적재한 스키마) 와 OpenAI 키만 있으면 됩니다.

In [ ]:
%pip install -q langgraph langchain langchain-openai sqlalchemy psycopg2-binary sqlparse pandas tabulate

In [ ]:
# Colab/로컬 환경에서 필요한 환경변수를 안전하게 로딩합니다 (다른 노트북과 동일 패턴).
import os

def _load_secret(key: str, required: bool = True) -> None:
    """Colab Secrets → getpass 입력 순으로 시도해 환경변수에 적재."""
    if os.environ.get(key):
        return
    value = None
    try:
        from google.colab import userdata  # type: ignore  (Colab 전용 모듈)
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        try:
            from getpass import getpass
            value = getpass(f"Enter {key}: ")
        except Exception:
            value = None
    if value:
        os.environ[key] = value
    elif required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets or via env var.")

# 이 노트북은 OpenAI(LLM) + Neon(DB) 두 키가 필수.
_load_secret("OPENAI_API_KEY", required=True)
_load_secret("NEON_DSN", required=True)
print("Environment ready.")

## 1. 에이전트 아키텍처

```
     질문
      │
      ▼
┌────────────────┐
│  generate_sql  │◄──────────────┐  (에러 피드백으로 재생성)
└────────┬───────┘               │
         │                       │
         ▼                       │
┌────────────────┐               │
│   execute_sql  │               │
└────────┬───────┘               │
         │                       │
         ▼                       │
┌────────────────┐               │
│  validate_sql  │               │
└────────┬───────┘               │
         │                       │
     should_retry?               │
       │    │                    │
  answer│    │retry (≤ 3회)      │
       ▼    └────────────────────┘
┌────────────────┐
│ generate_answer│
└────────┬───────┘
         ▼
       사용자
```

| 노드 | 입력 | 출력 | 역할 |
|---|---|---|---|
| `generate_sql` | `question`, (`error`, `sql`) | `sql`, `retry_count++` | 질문 → SQL. 이전 에러가 있으면 프롬프트에 포함. |
| `execute_sql` | `sql` | `result` or `error` | DB 실행. 예외는 `error` 로 캡처. |
| `validate_sql` | `sql`, `error` | `error` (정제) | 위험 키워드 차단. (실행은 execute 가 이미 했음) |
| `generate_answer` | `question`, `sql`, `result`, `error` | `answer` | 결과/에러를 한국어로 요약. |

**설계 결정 — validate 의 위치:**
- 교과서적 흐름은 `generate → validate → execute` 이지만, 본 구현은 **`execute` 를 먼저 시도** 하고 예외를 잡은 뒤 `validate` 에서 보안 키워드 검사를 하는 형태를 씁니다.
- 이유: LLM 이 생성한 SQL 의 **실질적 실패 원인** (존재하지 않는 컬럼, 잘못된 조인 등) 대부분은 DB 가 직접 던지는 에러로만 확인할 수 있기 때문입니다.
- **보안 차단** 은 `validate_sql` 이 수행하되, 실행 전에도 한 번 더 `is_safe_sql` 게이트를 둡니다 (`execute_sql` 내부).

## 2. Neon DB 엔진 연결

`NEON_DSN` 로 SQLAlchemy 엔진을 만듭니다. **읽기 전용 세션** 을 위해 `connect_args` 에 `default_transaction_read_only=on` 을 걸어 두면 LLM 이 어떤 SQL 을 만들어도 DB 수준에서 쓰기 작업을 거부합니다 (진정한 방어선).

> 실습 환경에서 읽기 전용 옵션이 거부되는 경우 (Neon 롤 설정 등) 주석의 fallback 으로 전환하세요.

In [ ]:
# DB 엔진 — 가능하면 "읽기 전용 세션" 으로 시작합니다.
# 이렇게 하면 LLM 이 어떤 위험 SQL 을 만들어도 DB 가 직접 거부 → 정규식 가드보다 강력한 진정한 방어선.
from sqlalchemy import create_engine, inspect, text
import pandas as pd

try:
    engine = create_engine(
        os.environ["NEON_DSN"],
        # `default_transaction_read_only=on` 은 PostgreSQL 세션 옵션 — 트랜잭션을 read-only 로 강제.
        connect_args={"options": "-c default_transaction_read_only=on"},
        pool_pre_ping=True,    # 끊긴 커넥션을 자동으로 다시 연결 (서버리스 Neon 권장 옵션)
    )
    # Smoke test: 단순 SELECT 한 번으로 연결 정상 여부 확인.
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("Engine connected in read-only mode.")
except Exception as e:
    # 일부 호스팅에서 read-only 옵션이 막혀 있을 수 있어 일반 모드로 폴백.
    print(f"[WARN] read-only 옵션 실패 → 일반 모드로 재연결: {e}")
    engine = create_engine(os.environ["NEON_DSN"], pool_pre_ping=True)
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("Engine connected (non-RO).")

## 3. 스키마 수집 — `collect_schema`

LLM 프롬프트에 넣을 **DDL 형태 스키마 텍스트** 를 자동으로 만듭니다. 컬럼 타입, NOT NULL, FK 관계, PostgreSQL `COMMENT ON COLUMN` 까지 싹 모읍니다.

> 학생이 본인 프로젝트로 전환할 때 가장 많이 막히는 부분 → 이 함수만 그대로 복사하고 `TABLES` 와 `engine` 만 바꾸면 끝.

In [ ]:
def collect_schema(engine, tables=None) -> str:
    """DB 스키마를 LLM 프롬프트용 DDL 텍스트로 변환."""
    inspector = inspect(engine)
    if tables is None:
        tables = inspector.get_table_names()

    parts = []
    for table in tables:
        columns = inspector.get_columns(table)
        fks = inspector.get_foreign_keys(table)

        col_lines = []
        for col in columns:
            nullable = "" if col["nullable"] else " NOT NULL"
            col_lines.append(f"    {col['name']} {col['type']}{nullable}")

        fk_lines = []
        for fk in fks:
            fk_lines.append(
                f"    FOREIGN KEY ({', '.join(fk['constrained_columns'])}) "
                f"REFERENCES {fk['referred_table']}({', '.join(fk['referred_columns'])})"
            )

        ddl = f"CREATE TABLE {table} (\n"
        ddl += ",\n".join(col_lines)
        if fk_lines:
            ddl += ",\n" + ",\n".join(fk_lines)
        ddl += "\n);"

        # PostgreSQL COMMENT 수집 (pg_attribute.attnum 기반)
        try:
            with engine.connect() as conn:
                comments = conn.execute(
                    text("""
                        SELECT a.attname,
                               col_description(c.oid, a.attnum) AS comment
                        FROM pg_class c
                        JOIN pg_namespace n ON n.oid = c.relnamespace
                        JOIN pg_attribute a ON a.attrelid = c.oid
                        WHERE c.relname = :table
                          AND n.nspname = 'public'
                          AND a.attnum > 0
                          AND NOT a.attisdropped
                        ORDER BY a.attnum
                    """),
                    {"table": table},
                ).fetchall()
            for col_name, comment in comments:
                if comment:
                    ddl += f"\n-- {table}.{col_name}: {comment}"
        except Exception:
            pass  # 코멘트 조회가 권한 문제 등으로 실패해도 DDL 본체는 유효

        parts.append(ddl)
    return "\n\n".join(parts)


# 병원 DB 기준 테이블 목록 — 본인 프로젝트에서는 이 리스트만 교체하세요.
TABLES = ["patients", "doctors", "visits", "diagnoses", "departments"]

# 실제 존재하는 테이블만 필터 (일부가 없어도 이 노트북이 계속 동작하도록)
_available = set(inspect(engine).get_table_names())
TABLES = [t for t in TABLES if t in _available]
if not TABLES:
    # 병원 DB 가 아직 없으면 현재 DB 의 모든 테이블을 사용
    TABLES = list(_available)[:10]
print(f"Tables to include in schema: {TABLES}")

SCHEMA = collect_schema(engine, TABLES)
print(f"\nSchema text length: {len(SCHEMA)} chars\n")
print(SCHEMA[:600] + ("..." if len(SCHEMA) > 600 else ""))

## 4. AgentState 정의

모든 노드가 공유하는 상태 스키마. `TypedDict` 로 **정적 타입 힌트** 만 주고, 초기값은 `invoke` 호출자가 명시적으로 넘겨야 합니다 (`attempts=0`, `history=[]` 등).

| 필드 | 타입 | 의미 |
|---|---|---|
| `question` | str | 사용자 질문 (원문) |
| `sql` | str | 현재 시도의 생성된 SQL |
| `result` | list (dict) | `execute_sql` 이 돌려준 rows — Markdown 변환은 `result_md` 에 별도 저장 |
| `result_md` | str | LLM 프롬프트에 넣을 Markdown 표 |
| `answer` | str | 최종 자연어 답변 |
| `error` | Optional[str] | 직전 단계 에러 메시지 (없으면 빈 문자열) |
| `retry_count` | int | 현재까지의 재시도 횟수 (최대 3) |

In [ ]:
# AgentState = "에이전트가 다루는 공유 메모(상태)" 의 형태를 정적 타입으로 선언.
# total=False → 모든 키를 매번 다 채울 필요는 없다 (부분 업데이트 허용).
from typing import TypedDict, Optional, List, Dict, Any
from langgraph.graph import StateGraph, END


class AgentState(TypedDict, total=False):
    question: str                  # 사용자 질문 (원문)
    sql: str                       # 현재 시도의 생성된 SQL (LIMIT 자동 주입 후 최종본)
    result: List[Dict[str, Any]]   # 실행 결과 (행 리스트)
    result_md: str                 # LLM 답변용 Markdown 표
    answer: str                    # 최종 자연어 답변
    error: Optional[str]           # 직전 단계 에러 메시지 (재시도 트리거)
    retry_count: int               # 재시도 횟수 (3회 도달 시 포기)


print("AgentState defined.")

## 5. 보안 가드레일 — `is_safe_sql` + LIMIT 주입

정규식 기반 블랙리스트는 **1 차 방어선** 입니다. 최종 방어는 **DB 수준의 read-only 세션** (위 엔진 설정) 이지만, LLM 이 위험 쿼리를 만들어 호출 비용만 낭비하는 것을 줄이기 위해 가드를 둡니다.

- 차단 키워드: `DROP`, `DELETE`, `UPDATE`, `INSERT`, `ALTER`, `TRUNCATE`, `CREATE`, `GRANT`, `REVOKE`.
- `LIMIT` 이 없는 SELECT 에는 `LIMIT 1000` 강제 주입.

> **한계 인지:** data-modifying CTE (`WITH x AS (DELETE ...) SELECT`), 함수 내부 쓰기, `COPY FROM PROGRAM` 등은 정규식으로 못 막습니다. read-only 세션이 필수입니다.

In [ ]:
import re

BLOCKED_SQL_PATTERN = re.compile(
    r"\b(DROP|DELETE|UPDATE|INSERT|ALTER|TRUNCATE|CREATE|GRANT|REVOKE)\b",
    re.IGNORECASE,
)


def is_safe_sql(sql: str) -> tuple[bool, str]:
    """(ok, reason). ok=False 면 reason 에 차단 사유."""
    match = BLOCKED_SQL_PATTERN.search(sql)
    if match:
        return False, f"보안 위반: '{match.group()}' 명령은 허용되지 않습니다."
    return True, ""


def inject_limit(sql: str, cap: int = 1000) -> str:
    """SELECT 에 LIMIT 이 없으면 끝에 LIMIT cap 추가. 이미 있으면 그대로."""
    stripped = sql.strip().rstrip(";")
    if re.search(r"\bLIMIT\s+\d+\b", stripped, re.IGNORECASE):
        return stripped
    return f"{stripped}\nLIMIT {cap}"


# 자가 테스트
for s in [
    "SELECT * FROM patients",
    "DROP TABLE visits",
    "WITH x AS (SELECT 1) SELECT * FROM x",
]:
    ok, reason = is_safe_sql(s)
    print(f"  is_safe_sql: ok={ok}, reason='{reason}' | {s[:50]}")
print()
print(inject_limit("SELECT * FROM patients"))

## 6. LLM 출력 클리닝 — Markdown 코드 펜스 제거

LLM 은 종종 SQL 을 ```` ```sql ... ``` ```` 로 감싸서 돌려줍니다. 실행 전 펜스를 제거해야 합니다.

In [ ]:
def strip_sql_fences(sql: str) -> str:
    """```sql ... ``` 또는 ``` ... ``` 펜스 제거."""
    sql = re.sub(r"```sql\s*", "", sql, flags=re.IGNORECASE)
    sql = re.sub(r"```\s*", "", sql)
    return sql.strip()


# 자가 테스트
for s in [
    "```sql\nSELECT 1;\n```",
    "```\nSELECT 2;\n```",
    "SELECT 3;",
]:
    print(repr(strip_sql_fences(s)))

## 7. LLM 인스턴스 — SQL 생성용 + 답변 생성용

비용 대비 성능 균형으로 `gpt-4o-mini` 를 기본으로 씁니다. `temperature=0` 은 **결정적 SQL 생성** 에 필수.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

sql_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
answer_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

print("LLMs ready.")

## 8. 노드 1 — `generate_sql`

프롬프트 설계 포인트:

1. **스키마** (`{schema}`) 를 통째로 주입.
2. **비즈니스 규칙** (`visits.status='completed'`, 나이 계산법 등) — 본인 프로젝트에서 여기를 꼭 바꾸세요.
3. **에러 피드백** — 이전 시도가 실패했다면 "이 SQL 이 이 에러로 터졌다. 같은 실수 하지 말 것." 을 프롬프트에 추가 → 재시도 루프의 학습 효과.
4. **SQL 만 반환** 지시 + 펜스 제거로 출력 정제.

In [ ]:
# 노드 1: generate_sql — 자연어 질문 → SQL.
# 핵심 트릭: 직전 시도가 실패했다면 "그 SQL과 그 에러 메시지" 를 다음 프롬프트에 주입.
# 이렇게 해야 LLM 이 같은 실수를 반복하지 않습니다 (= 재시도 루프의 학습 효과).
SQL_GEN_TEMPLATE = ChatPromptTemplate.from_template(
    """당신은 PostgreSQL 전문가입니다. 아래 스키마를 참고하여 질문에 대한 SQL 하나를 작성하세요.

## 스키마
{schema}

## 규칙 (본인 프로젝트에서는 이 블록을 재정의하세요)
- SELECT 문만 작성. DML/DDL 금지.
- visits.status = 'completed' 만 유효한 진료로 간주.
- 나이 = EXTRACT(YEAR FROM AGE(birth_date))
- 결과가 많을 가능성이 있으면 LIMIT 100 이하를 권장.
- 설명 없이 **SQL 만** 반환.
{error_feedback}

## 질문
{question}

SQL:
"""
)

# LCEL 파이프 — 프롬프트 → LLM → 문자열 파서 한 줄로 연결 (12번 노트북 참조).
sql_gen_chain = SQL_GEN_TEMPLATE | sql_llm | StrOutputParser()


def generate_sql(state: AgentState) -> dict:
    """질문 → SQL. 이전 에러가 있으면 피드백으로 재생성."""
    # 1) 직전 에러가 있다면 프롬프트에 끼워 LLM 이 그 실수를 피하도록 한다.
    error_feedback = ""
    if state.get("error"):
        error_feedback = (
            "\n## 직전 시도의 실패\n"
            f"- 실패 SQL:\n{state.get('sql', '')}\n"
            f"- 에러: {state['error']}\n"
            "이 에러를 피해서 SQL 을 다시 작성하세요."
        )

    # 2) 체인을 호출 — 변수 3개를 dict 로 한 번에 전달.
    raw = sql_gen_chain.invoke({
        "schema": SCHEMA,
        "question": state["question"],
        "error_feedback": error_feedback,
    })
    sql = strip_sql_fences(raw)  # ```sql ... ``` 펜스 제거

    # 3) state 에 덮어쓸 키만 반환 — LangGraph 가 기존 state 와 자동 병합.
    return {
        "sql": sql,
        "retry_count": state.get("retry_count", 0) + 1,
        # error 는 여기서 초기화하지 않음 — execute_sql 이 성공하면 자연스레 빈 문자열로 덮음.
    }

## 9. 노드 2 — `execute_sql`

- 실행 전에 `is_safe_sql` 게이트 → 차단 시 `error` 기록하고 바로 반환.
- LIMIT 강제 주입 → `inject_limit`.
- `pandas.read_sql(text(sql), engine)` 으로 실행. 예외는 모두 `error` 로 캡처.
- 결과가 비어 있는 경우 (`result_md = "(결과 없음)"`) 는 **에러가 아님** — 검증 노드에서 구분합니다.

In [ ]:
# 노드 2: execute_sql — 실제 DB 실행 + 에러 캡처.
def execute_sql(state: AgentState) -> dict:
    sql = state.get("sql", "")
    if not sql:
        # 보호 장치: 빈 SQL 이 들어오면 즉시 에러 처리하고 반환.
        return {"error": "실행할 SQL 이 없습니다.", "result": [], "result_md": ""}

    # 1차 정규식 가드 — 영문 DML/DDL 키워드를 차단 (보조 방어선).
    ok, reason = is_safe_sql(sql)
    if not ok:
        return {"error": reason, "result": [], "result_md": ""}

    # LIMIT 자동 주입 — 학생이 LIMIT 없이 SELECT 를 던져도 1000행에서 자르도록 안전망.
    safe_sql = inject_limit(sql, cap=1000)

    try:
        # text() 로 감싸야 SQLAlchemy 가 % 같은 특수문자를 안전하게 처리합니다 (01번 issue 참조).
        df = pd.read_sql(text(safe_sql), engine)
        if df.empty:
            # 결과가 비어 있는 것은 "에러" 가 아니라 "조건에 맞는 데이터 없음" 의 정상 상태.
            return {
                "result": [],
                "result_md": "(결과 없음 — 조건을 다시 확인하세요)",
                "error": "",
                "sql": safe_sql,   # LIMIT 주입된 최종 SQL 을 state 에 반영
            }
        # 결과를 두 형태로 보관:
        #  (1) result_rows  : dict 리스트 — 추후 프로그램적 후처리에 편함.
        #  (2) md_table     : Markdown 표 — LLM 답변 프롬프트에 그대로 넣기 좋음.
        # head(50) 으로 잘라 LLM 컨텍스트를 아끼고 토큰 비용 통제.
        result_rows = df.head(50).to_dict(orient="records")
        md_table = df.head(50).to_markdown(index=False)
        if len(df) > 50:
            md_table += f"\n\n... 외 {len(df) - 50}행"
        return {
            "result": result_rows,
            "result_md": md_table,
            "error": "",
            "sql": safe_sql,
        }
    except Exception as e:
        # 어떤 예외가 나든 (구문 오류·존재하지 않는 컬럼·권한 문제 등)
        # 에러 메시지를 캡처해 다음 generate_sql 의 프롬프트에 피드백 입력으로 재사용한다.
        return {
            "error": f"SQL 실행 오류: {type(e).__name__}: {str(e)[:300]}",
            "result": [],
            "result_md": "",
        }

## 10. 노드 3 — `validate_sql`

`execute_sql` 이 이미 실행+예외 캡처를 했으므로 여기서 할 일은:
- `error` 가 이미 있으면 **그대로 통과** → 다음 분기에서 재시도/포기 결정.
- `error` 가 없으면 **깨끗한 상태** (`error=""`) 를 돌려보내 `generate_answer` 로.
- 빈 결과 (`result_md="(결과 없음 ...)`) 는 에러가 **아님** — 사용자에게 "해당 조건에 맞는 데이터가 없음" 으로 정직하게 전달.

In [ ]:
def validate_sql(state: AgentState) -> dict:
    # 이미 에러가 있으면 분기 함수가 retry 를 결정할 것
    if state.get("error"):
        return {}  # 아무것도 덮어쓰지 않음
    # 성공 경로 → error 를 명시적으로 비워 둠
    return {"error": ""}

## 11. 노드 4 — `generate_answer`

- 재시도 3 회 소진 + 에러 지속 시: 사과 메시지 + 마지막 에러를 함께 반환 (투명성 ↑).
- 정상 경로: SQL + 결과 Markdown 을 LLM 에 주고 **한국어 2~3 문장 요약** 지시.
- 결과가 비었으면 "해당 조건에 맞는 데이터가 없습니다" 를 솔직히 안내.

In [ ]:
ANSWER_TEMPLATE = ChatPromptTemplate.from_template(
    """다음 SQL 실행 결과를 바탕으로 질문에 한국어로 답변하세요.

## 질문
{question}

## 실행한 SQL
{sql}

## 결과 (Markdown 표)
{result_md}

## 규칙
- 2-3 문장으로 핵심만 요약.
- 숫자에 천 단위 구분자(쉼표) 사용.
- 결과가 비어 있으면 "해당 조건에 맞는 데이터가 없습니다." 로 시작하는 안내.
- 추측 금지 — 표에 없는 수치는 언급하지 말 것.
"""
)

answer_chain = ANSWER_TEMPLATE | answer_llm | StrOutputParser()


def generate_answer(state: AgentState) -> dict:
    # 재시도 소진 + 에러 남아 있음 → 사과 메시지
    if state.get("error") and state.get("retry_count", 0) >= 3:
        return {
            "answer": (
                "죄송합니다. 질문에 답변하지 못했습니다.\n"
                f"- 마지막 에러: {state['error']}\n"
                "- 질문을 더 구체적으로 다시 물어봐 주세요."
            )
        }

    ans = answer_chain.invoke({
        "question": state.get("question", ""),
        "sql": state.get("sql", ""),
        "result_md": (state.get("result_md") or "(결과 없음)")[:1500],
    })
    return {"answer": ans.strip()}

## 12. 분기 함수 — `should_retry`

- `error` 비어 있음 → `answer` 로 직진.
- `error` 있고 `retry_count < 3` → `retry` (= `generate_sql` 로 되돌림).
- `error` 있고 `retry_count >= 3` → `giveup` (= `generate_answer` 로 가되 사과 메시지).

**라벨 / path_map 키 일치가 필수입니다.** 불일치 시 LangGraph 가 런타임 에러를 던집니다.

In [ ]:
MAX_RETRIES = 3


def should_retry(state: AgentState) -> str:
    if not state.get("error"):
        return "answer"
    if state.get("retry_count", 0) >= MAX_RETRIES:
        return "giveup"
    return "retry"


# 자가 테스트
for s in [
    {"error": "", "retry_count": 1},
    {"error": "syntax error", "retry_count": 1},
    {"error": "syntax error", "retry_count": 3},
]:
    print(f"  {s} → {should_retry(s)}")

## 13. 그래프 조립 + 컴파일

```
[generate_sql] → [execute_sql] → [validate_sql] → (condition) → [generate_answer] → END
      ▲                                                │
      └────────────────── retry ───────────────────────┘
```

In [ ]:
# 그래프 조립 — 4 노드 등록 + 직선 엣지 + 조건부 분기 한 번.
# 이 한 셀이 16번 노트북에서 익힌 "StateGraph 5단계" 의 실전 적용입니다.
graph = StateGraph(AgentState)

# ① 노드 등록 (이름은 자유롭게, 함수 이름과 같게 하면 디버깅이 쉬움)
graph.add_node("generate_sql", generate_sql)
graph.add_node("execute_sql", execute_sql)
graph.add_node("validate_sql", validate_sql)
graph.add_node("generate_answer", generate_answer)

# ② 시작점 + 직선 엣지
graph.set_entry_point("generate_sql")
graph.add_edge("generate_sql", "execute_sql")
graph.add_edge("execute_sql", "validate_sql")

# ③ 조건부 분기 — should_retry 의 반환 라벨에 따라 다음 노드가 동적으로 결정됨.
# path_map 의 키("answer"/"giveup"/"retry") 는 should_retry 가 반환하는 문자열과 정확히 일치해야 합니다.
graph.add_conditional_edges(
    "validate_sql",
    should_retry,
    {
        "answer": "generate_answer",
        "giveup": "generate_answer",   # 같은 노드 — 함수 내부에서 사과 메시지 분기
        "retry":  "generate_sql",      # 핵심 ★ 자기 자신 쪽으로 되돌아가는 루프
    },
)
graph.add_edge("generate_answer", END)

# ④ .compile() 로 빌더에서 실제 실행 가능한 객체 생성.
agent = graph.compile()
print("Agent compiled.")

In [ ]:
from IPython.display import Image, display

try:
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"[info] mermaid png 렌더 실패 ({e}) → 텍스트로 출력:")
    print(agent.get_graph().draw_mermaid())

## 14. 실행 도우미 — `ask_agent(question, verbose=True)`

- `verbose=True` (기본): `agent.stream(...)` 으로 **각 노드 실행 이벤트**를 찍음. 재시도가 일어나면 `generate_sql` 이 여러 번 등장하는 것을 눈으로 확인할 수 있음.
- `verbose=False`: `agent.invoke(...)` 로 최종 결과만.

초기 상태에서 `retry_count=0`, `error=""` 를 **명시적 초기화** 하는 점을 강조하세요 — TypedDict 에는 기본값이 없습니다.

In [ ]:
# 실행 도우미 — 두 가지 호출 모드를 한 함수로 노출.
def _initial_state(question: str) -> dict:
    """모든 키를 빈 값으로 초기화 — TypedDict 에는 기본값이 없어 호출자가 명시적으로 채워야 한다."""
    return {
        "question": question,
        "sql": "",
        "result": [],
        "result_md": "",
        "answer": "",
        "error": "",
        "retry_count": 0,
    }


def ask_agent(question: str, verbose: bool = True) -> dict:
    """에이전트 호출 + 상태 전이 추적. 최종 state dict 반환."""
    print(f"\n{'='*60}\n❓ {question}\n{'='*60}")

    if not verbose:
        # invoke 모드 — 그래프 끝까지 한 번에 실행, 최종 state 만 반환.
        state = agent.invoke(_initial_state(question))
        print(f"\n📝 SQL (최종):\n{state.get('sql', '')}")
        print(f"\n💬 답변: {state.get('answer', '')}")
        return state

    # stream 모드 — 노드 하나가 끝날 때마다 이벤트가 yield 되어 들어옴.
    # 재시도가 발생하면 generate_sql 노드 이름이 여러 번 찍히는 게 눈으로 보입니다 (= 데모 효과).
    last = {}
    for event in agent.stream(_initial_state(question)):
        # event 는 {노드명: 그 노드가 반환한 dict}. 보통 키가 1개이지만 일반 형태로 순회.
        for node_name, output in event.items():
            last = {**last, **(output or {})}  # dict 병합 — 누적 state 만들기
            print(f"\n📍 [{node_name}]")
            # 각 노드가 채운 키를 골라 한 줄씩 요약 출력
            if output.get("sql"):
                sql_preview = output['sql'].replace("\n", " ")
                print(f"   SQL: {sql_preview[:120]}{'...' if len(sql_preview) > 120 else ''}")
            if output.get("error"):
                print(f"   ❌ Error: {output['error'][:120]}")
            if output.get("result_md"):
                first_line = output['result_md'].split("\n")[0][:80]
                print(f"   ✅ Result: {first_line} ...")
            if output.get("answer"):
                print(f"   💬 Answer: {output['answer'][:200]}")
    return last

## 15. 시연 1 — 단일 질문 3 개

세 가지 질문을 돌려 **정상 경로** 와 **재시도 경로** 를 모두 체감합니다.

1. **쉬운 집계** — 단번에 성공 (`generate_sql` 1 회).
2. **조인이 필요한 질문** — 보통 1~2 회 재시도에 성공.
3. **의도적으로 어려운 질문** (모호한 표현) — 재시도 로그가 드러나도록 유도.

> 만약 강의 시연에서 재시도가 전혀 나오지 않으면 `ask_agent` 를 몇 차례 더 호출하거나, 학생에게 질문을 바꿔 보게 하세요. 재시도 로직이 **존재하지만 발동하지 않는** 것 자체는 에이전트가 잘 동작한다는 신호입니다.

In [ ]:
# 시연 1 — 쉬운 질문
_ = ask_agent("현재 등록된 환자 수는 몇 명인가요?")

In [ ]:
# 시연 2 — 조인이 필요한 질문
_ = ask_agent("진료과별 의사 수를 많은 순서대로 보여주세요.")

In [ ]:
# 시연 3 — 모호 / 복합 질문 (재시도 가능성이 높음)
_ = ask_agent("최근 3개월간 가장 자주 방문한 환자 Top 5 와 해당 환자의 주요 진료과를 함께 보여주세요.")

## 16. 시연 2 — 재시도 강제 발동 데모

위에서 재시도가 한 번도 안 일어났다면? **존재하지 않는 컬럼을 LLM 에게 초기 조건으로 넣어** 재시도 루프를 강제 발동시킬 수 있습니다.

아래 셀은 `generate_sql` 을 건너뛰고 **의도적으로 잘못된 SQL** 로 시작 상태를 주입 → `execute_sql` 에서 DB 에러 → `validate_sql` → 분기가 `retry` 를 리턴 → `generate_sql` 이 에러 피드백을 받아 **올바른 SQL 로 재생성** 하는 과정을 보여줍니다.

> 이 데모는 `agent.invoke(state_with_broken_sql_and_retry_count_0)` 형태로, 상태 일부를 미리 채워 넣는 **디버깅 트릭**입니다. 실제 프로덕션 호출은 `_initial_state(question)` 을 그대로 씁니다.

In [ ]:
# 재시도 강제 데모
# 1) 잘못된 SQL 을 초기 상태로 심고 retry_count=0 으로 시작
# 2) execute_sql 이 에러 → validate_sql → 분기 retry → generate_sql 재생성
broken_state = {
    "question": "전체 환자 수는?",
    "sql": "SELECT COUNT(*) FROM nonexistent_table_xyz",
    "result": [],
    "result_md": "",
    "answer": "",
    "error": "",
    "retry_count": 0,
}

# 진입점이 generate_sql 이므로, 잘못된 SQL 을 강제로 실행시키려면
# execute_sql 부터 들어가야 함 → 별도 서브그래프로 흉내내는 대신
# 단순히 첫 invoke 후 agent.stream 으로 상태 전이를 보여줍니다.

# 더 확실한 데모: 질문 자체에 "존재하지 않는 테이블/컬럼" 힌트를 넣어
# LLM 이 착각하도록 유도
trick_q = (
    "비활성 환자 목록을 is_active=false 컬럼 기준으로 조회해 주세요."
    "  (참고: 실제 스키마에 is_active 컬럼이 없을 가능성이 높습니다.)"
)
result = ask_agent(trick_q)
print(f"\n📊 최종 retry_count = {result.get('retry_count', 0)}")
print(f"📊 최종 error       = '{result.get('error', '')[:100]}'")

## 17. 10 개 질문 일괄 테스트

과제 #3 의 합격 기준과 동일한 형식으로 **10 개 질문 × 1 회씩** 돌립니다. 각 질문당 `retry_count`, 에러 유무, SQL 미리보기, 답변을 표로 정리합니다.

> **합격 기준:** 10 개 중 **7 개 이상** `error=""` + 비어 있지 않은 `answer`.
> 본인 프로젝트 질문 10 개로 이 셀을 돌렸을 때 7/10 이상이 나오면 과제 #3 제출 가능합니다.

In [ ]:
# 10 개 질문 일괄 테스트 — 과제 #3 합격 기준(7/10 이상 ✅) 과 동일한 형식.
project_questions = [
    "전체 환자 수는?",
    "남성 환자 중 40세 이상은 몇 명?",
    "진료과별 의사 수를 보여줘",
    "지난달 완료 진료 건수는?",
    "응급 진료 평균 비용은?",
    "가장 많이 방문한 환자 Top 3는?",
    "중증 진단을 받은 환자 이름은?",
    "2026년 월별 방문 수 추이는?",
    "내과 의사 중 급여 최고는?",
    "혈액형별 환자 분포는?",
]

rows = []
for q in project_questions:
    # invoke = 끝까지 한 번에 실행. (각 노드 추적이 필요하면 ask_agent 사용)
    state = agent.invoke(_initial_state(q))
    # 합격 판정: 답변이 비어 있지 않고 에러도 없어야 ✅
    ok = bool(state.get("answer")) and not state.get("error")
    rows.append({
        "question": q,
        "status": "✅" if ok else "❌",
        "retries": state.get("retry_count", 0),   # 시도 횟수 — 재시도 발동 여부의 단서
        # 결과 표가 너무 옆으로 길어지지 않도록 줄바꿈 제거 + 80자에서 자르기
        "sql": (state.get("sql") or "").replace("\n", " ")[:80],
        "answer": (state.get("answer") or "").replace("\n", " ")[:80],
    })
    print(f"{rows[-1]['status']} [{rows[-1]['retries']}회] {q}")

df_test = pd.DataFrame(rows)
success = (df_test["status"] == "✅").sum()
print(f"\n🎯 정답률: {success}/{len(project_questions)} ({success/len(project_questions)*100:.0f}%)")
df_test

## 실습 과제

다음 1 가지 실습을 직접 작성해 보세요. (정답 코드는 의도적으로 비워 두었습니다.)

### 1. 에이전트 실행 추적
`verbose=True`로 실행하면 각 노드를 거치는 과정이 출력됩니다:

```
============================================================
❓ 현재 등록된 환자 수는 몇 명인가요?
============================================================

📍 [generate_sql]
   SQL: SELECT COUNT(*) AS total_patients FROM patients...

📍 [run_sql]
   (SQL 실행 성공)

📍 [validate]
   (에러 없음 --> answer로 이동)

📍 [answer]
   💬 Answer: 현재 등록된 환자는 총 150명입니다.
```

에러가 발생하면 재시도 과정도 추적됩니다:

```
📍 [generate_sql]    (1차 시도)
   SQL: SELECT ... FROM pateints ...  (오타!)

📍 [run_sql]
   ❌ Error: relation "pateints" does not exist

📍 [validate]
   (에러 있음, attempts=1 --> generate_sql로 재시도)

📍 [generate_sql]    (2차 시도, 에러 피드백 포함)
   SQL: SELECT ... FROM patients ...  (수정됨!)

📍 [run_sql]
   (SQL 실행 성공)

📍 [validate]
   (에러 없음 --> answer로 이동)

📍 [answer]
   💬 Answer: ...
```

_힌트: 위 노트북에서 정의한 `ask_agent(question, verbose=True)` 를 호출하고, 각 노드 이벤트가 위와 비슷한 형식으로 찍히는지 확인하세요. 재시도가 일어나는 모호한 질문도 한 번 시도해 본인 에이전트의 추적 로그를 캡처해 두세요._


In [ ]:
# ============================================================
# 실습 과제 — 에이전트 실행 추적 (verbose=True)
# ============================================================

# 실습 1: ask_agent 를 verbose=True 로 호출해 노드별 이벤트 로그를 확인
# TODO: ask_agent("...", verbose=True) 를 정상 질문 1개와 재시도가 발생할 만한 모호 질문 1개로 각각 호출하고 generate_sql / run_sql / validate / answer 로그를 직접 캡처해 보세요.
# 여기에 구현하세요.


## 19. 과제 #3 — 에이전트 v1 (Day 4 시작까지 제출)

```
╔════════════════════════════════════════════════════════╗
║                과제 #3 — 에이전트 v1                     ║
╠════════════════════════════════════════════════════════╣
║                                                        ║
║  제출 기한: Day 4 시작 (21H)                            ║
║                                                        ║
║  제출물:                                                ║
║  1. Colab 노트북: <이름>_sql_agent.ipynb                 ║
║     (이 17 번 노트북을 본인 DB/규칙으로 포크)             ║
║  2. 본인 도메인 질문 10 개 일괄 테스트 결과표              ║
║  3. LangSmith trace URL (Day 4 21H 에서 연결)           ║
║                                                        ║
║  합격 기준: 10 개 중 7 개 이상 ✅                         ║
║                                                        ║
║  💡 오늘 밤 본인 프로젝트 DB 로 에이전트를 먼저 돌려       ║
║     두면 내일 LangSmith 연결·Ragas 평가가 훨씬 수월      ║
║     합니다.                                             ║
║                                                        ║
╚════════════════════════════════════════════════════════╝
```

## 20. 이 노트북의 의미 — 다시 한 번

이것이 **과정 첫 시간에 보여드린 "00 데모 에이전트" 의 본체** 입니다. Day 1 첫 시간에는 완성된 에이전트가 질문을 받으면 답변을 내놓는 모습만 보았지만, 이제 여러분은 그 내부의 **4 개 노드, 조건부 분기, 재시도 루프, 프롬프트 피드백** 을 직접 조립했습니다.

- 과제 #3 를 제출하고 나면 → Day 4 21H 에서 **LangSmith** 를 붙여 모든 호출을 자동 기록.
- Day 4 22H 에서 **Ragas** 로 **Faithfulness / Answer Relevancy / Context Precision·Recall** 을 정량 측정.
- 낮은 점수 질문을 진단 → 프롬프트 튜닝 v2 → 재평가.
- 23H 발표 리허설 → 24H 5~7 분 발표 + 수료.

수고하셨습니다.

## 다음 노트북에서는…

**`18_langsmith_tracing.ipynb`** (Day 4 21H) — 이 에이전트에 **LangSmith** 를 붙여 모든 노드 실행을 자동으로 트레이스로 기록하고, **토큰/지연/비용** 을 API 로 분석합니다. 본 노트북의 `agent` 코드가 거의 그대로 재사용되니, 17 번이 잘 돌아가면 18 번은 환경변수 한 줄 추가로 끝납니다.